# Week 6 &mdash; Final Project

## Shape Segmentation on 3D Mesh Data using Graph Neural Networks

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

This capstone synthesises the entire course. We frame **3D mesh segmentation** &mdash; assigning a semantic label to every vertex of a surface &mdash; as a **node-classification problem on a graph**, attacked with the message-passing networks of Week 3, using the intrinsic geometric features of Week 5.

This notebook is the **project brief and methodology**. The companion `implementation.ipynb` contains the full, runnable pipeline.

---


## 1. Problem Statement

Given a triangle mesh $\mathcal{M} = (V, E, F)$, **shape segmentation** seeks a label map
$$
y : V \to \{1, \dots, C\}
$$
assigning each vertex to one of $C$ semantic parts (e.g. for a human body: *head, torso, arms, legs*; for a chair: *seat, back, legs, armrests*).

This is a **per-vertex prediction**, so the task demands **permutation-equivariance** (relabelling vertices relabels predictions identically) rather than mere invariance &mdash; precisely the property of message-passing layers established in Week 3. Where Week 3 classified *nodes of an abstract graph*, here the graph is the **mesh connectivity** and the node features are **intrinsic geometric descriptors**.


## 2. How the Course Comes Together

This project is the GDL blueprint (Week 1) instantiated end to end:

- **Domain & symmetry (Week 1).** The domain is the mesh surface; the relevant symmetry is *isometry* (non-rigid deformation) and vertex *permutation*. We want features invariant to isometry and a network equivariant to permutation.
- **Graph & Laplacian (Week 2).** The mesh is a weighted graph; the cotangent Laplacian provides its geometry-aware connectivity and spectral basis.
- **Message passing (Week 3).** A GCN/GAT propagates and refines per-vertex features along mesh edges, producing permutation-equivariant per-vertex logits.
- **3D & invariance (Week 4).** Raw coordinates are *not* isometry-invariant; we must engineer invariant input features, echoing the rotation problem of PointNet.
- **Riemannian geometry (Week 5).** The **Heat Kernel Signature** and Laplace&ndash;Beltrami eigenfunctions supply isometry-invariant, multi-scale per-vertex descriptors &mdash; the ideal node features.

The result is a model that segments deformable shapes consistently regardless of pose &mdash; the payoff of building in the right geometric priors.


## 3. Methodology

**Pipeline.**

1. **Graph construction.** Convert the mesh into a graph whose nodes are vertices and whose edges are mesh edges. Edge weights may use the cotangent weights of Week 5.
2. **Feature engineering.** Compute *isometry-invariant* per-vertex descriptors:
   - **Heat Kernel Signature (HKS)** at several time scales (Week 5).
   - **Laplace&ndash;Beltrami eigenfunctions** $\phi_1, \dots, \phi_k$ as positional encodings.
   - Local geometric quantities (e.g. an estimate of Gaussian/mean curvature).
   All are intrinsic, hence invariant to rigid and (approximately) non-rigid deformation.
3. **Model.** A stack of message-passing layers (GCN or GAT) maps node features to per-vertex class logits.
4. **Training.** Minimise per-vertex cross-entropy with a train/validation split over meshes or vertices.
5. **Evaluation.** Report per-vertex **accuracy** and **mean Intersection-over-Union (mIoU)**, the standard segmentation metric.

**Why intrinsic features matter.** If we fed raw $(x,y,z)$ coordinates, the model would tie predictions to a canonical pose and fail on deformed inputs &mdash; the exact failure mode demonstrated for PointNet under rotation in Week 4. Intrinsic spectral features sidestep this by construction.


## 4. Evaluation Metrics

For predicted labels $\hat{y}$ and ground truth $y$ over $|V|$ vertices:

**Per-vertex accuracy.**
$$
\mathrm{Acc} = \frac{1}{|V|}\sum_{i} \mathbf{1}[\hat{y}_i = y_i].
$$

**Mean Intersection-over-Union.** For each class $c$,
$$
\mathrm{IoU}_c = \frac{|\{i: \hat{y}_i = c\} \cap \{i: y_i = c\}|}{|\{i: \hat{y}_i = c\} \cup \{i: y_i = c\}|},
\qquad
\mathrm{mIoU} = \frac{1}{C}\sum_{c=1}^{C} \mathrm{IoU}_c.
$$
mIoU is preferred over raw accuracy because it is not dominated by large parts and penalises both false positives and false negatives per class.


## 5. Datasets and Tooling

**Standard benchmarks.** Real research uses datasets such as the *Princeton Segmentation Benchmark*, *ShapeNet-Part*, *COSEG*, or *Human Body Segmentation*. These provide meshes with ground-truth per-vertex/face labels.

**This course.** The companion implementation generates a **synthetic, fully labelled mesh dataset** so the pipeline is self-contained and reproducible without large downloads. The methodology transfers directly: swap the synthetic loader for a real dataset loader and the rest of the pipeline is unchanged.

**Tooling.** PyTorch Geometric (`torch_geometric.nn` for the GNN layers, `torch_geometric.data.Data` for graphs), with SciPy for the spectral feature computation from Week 5.


## 6. Deliverables and Extensions

**Deliverables.**

- A trained GNN that segments meshes into semantic parts.
- Quantitative results: per-vertex accuracy and mIoU on a held-out split.
- A qualitative visualisation of predicted vs. ground-truth segmentations.
- A short discussion of failure cases and the role of feature choice.

**Suggested extensions.**

1. **Architecture comparison.** GCN vs. GAT vs. GraphSAGE; effect of depth (watch for over-smoothing, Week 3).
2. **Feature ablation.** Train with raw coordinates only, then add HKS, then add eigenfunction positional encodings; quantify the gain from intrinsic features.
3. **Isometry robustness.** Evaluate on deformed/rotated copies of test meshes and confirm intrinsic features preserve performance &mdash; closing the loop with Week 4.
4. **Equivariant baseline.** Replace the GNN with an $SE(3)$-equivariant point network (`e3nn`) and compare.

### References

- Hamilton, W. L. (2020). *Graph Representation Learning.* (Node classification, GNN architectures.)
- Bronstein, M. M., et al. (2021). *Geometric Deep Learning.* arXiv:2104.13478.
- Hanocka, R., et al. (2019). *MeshCNN: A Network with an Edge.* SIGGRAPH.
- Qi, C. R., et al. (2017). *PointNet.* CVPR. (Segmentation head design.)

Proceed to `implementation.ipynb` for the complete, runnable project.
